# TP Integrador 2 — RAG local sobre protección de datos personales

Procesamiento de Lenguaje Natural — Tecnicatura Superior en Ciencias de Datos e Inteligencia Artificial (IFTS N.º 24).

---

## Objetivo

Este sistema permite que una persona responsable de un proyecto de datos, en un organismo público o en una empresa, consulte qué debe cumplir y qué se recomienda cuando su proyecto trata datos personales o usa IA (por ejemplo, cuándo necesita el consentimiento de los titulares, en qué casos conviene una evaluación de impacto o qué exigirle a un proveedor de nube), en la Ley 25.326 y en dos guías oficiales de la AAIP, para diseñar sus proyectos cumpliendo la normativa antes de ponerlos en marcha.

La Ley 25.326 es obligatoria; la Guía de la AAIP y los Lineamientos son recomendaciones.

## Producto esperado

Un índice Chroma local, una tabla de pruebas en `pruebas/auditoria.csv` y una interfaz Gradio que muestra la respuesta junto con los fragmentos enviados al modelo.

## Preguntas de prueba

Escritas antes de evaluar: 3 respondibles (R1–R3, una por documento), 2 sin respuesta (S1–S2) y 1 caso de limitación (L1), cuyo dato está en el PDF pero dentro de una imagen.

| # | Pregunta | Fuente esperada |
|---|---|---|
| R1 | ¿En qué casos mi organización debería hacer una evaluación de impacto en protección de datos antes de desarrollar un sistema de IA? | Guía, p. 29 |
| R2 | ¿Qué debo tener en cuenta al contratar un proveedor de nube para alojar datos personales? | Lineamientos, p. 27 |
| R3 | ¿Necesito el consentimiento de las personas para usar sus datos personales en mi proyecto? ¿Hay excepciones? | Ley 25.326, art. 5, p. 4 |
| S1 | ¿Cuánto tarda la AAIP en aprobar una evaluación de impacto? | No figura: debería abstenerse |
| S2 | ¿Cuánto cuesta inscribir una base de datos en el Registro Nacional de Bases de Datos? | No figura: debería abstenerse (trampa: la Ley dice que la consulta del registro es gratuita, art. 13, p. 7) |
| L1 | ¿Qué valor de riesgo corresponde a una probabilidad alta con un impacto crítico? | Guía, p. 30 (Figura 3): el dato está en una imagen que no se extrae; debería abstenerse |

## 1. Preparar el entorno y declarar los documentos

Los tres PDF están en `datos/` y se publican con el repositorio (ver `datos/LEEME.md`). Descargados el 13/11/2026.

**Guía de IA responsable (AAIP)**

| Campo | Dato |
|---|---|
| Fuente | Guía para entidades públicas y privadas en materia de Transparencia y Protección de Datos Personales para una Inteligencia Artificial responsable — AAIP, versión preliminar, junio de 2024. `guia_ia_responsable_aaip_2024.pdf` |
| Recorte | Páginas impresas 6 a 49 |
| Unidad de análisis | Una página del PDF, luego un fragmento con `source` y `page`. |
| Transformación previa | Ninguna: PDF original del organismo. |
| Límite | Versión preliminar; repite un encabezado propio en cada página (no se limpió). |
| Uso y redistribución | Creative Commons Atribución 4.0 (términos de argentina.gob.ar). |

**Lineamientos del Plan de Protección de Datos**

| Campo | Dato |
|---|---|
| Fuente | Lineamientos para la formulación de un Plan de Protección de Datos Personales — Consejo Federal para la Transparencia, editado por la AAIP, 2023. `lineamientos_plan_datos_cft_aaip_2023.pdf` |
| Recorte | Páginas impresas 9 a 30 |
| Unidad de análisis | Una página del PDF, luego un fragmento con `source` y `page`. |
| Transformación previa | Ninguna: PDF original del organismo. |
| Límite | Gráficos, columnas y saltos de línea pueden extraerse de forma incompleta. |
| Uso y redistribución | Creative Commons Atribución 4.0 (términos de argentina.gob.ar). |

**Ley 25.326**

| Campo | Dato |
|---|---|
| Fuente | Ley 25.326 de Protección de los Datos Personales, texto actualizado (incluye Ley 26.343, BO 9/1/2008) — Congreso de la Nación. `ley_25326_texto_actualizado.pdf` |
| Recorte | Páginas impresas 1 a 20 (la 21 solo tiene el pie) |
| Unidad de análisis | Una página del PDF, luego un fragmento con `source` y `page`. |
| Transformación previa | Página web de argentina.gob.ar guardada como PDF desde el navegador el 13/11/2026. El PDF nativo de la HCDN no estaba disponible (error 503). |
| Límite | Cada página conserva encabezado y pie del navegador (fecha, título, URL, «n/21»). No se limpió. |
| Uso y redistribución | Norma pública; Creative Commons Atribución 4.0 (términos de argentina.gob.ar). |

In [3]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(".env")
print("Librerías cargadas.")

C:\Users\josel\AppData\Local\Temp\ipykernel_5312\2314591115.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Librerías cargadas.


## 2. Elegir un recorte que puedas controlar

De cada documento se indexa solo el cuerpo: Guía AAIP pp. 6–49 (de la Introducción a las Consideraciones finales), Lineamientos pp. 9–30 (de la Introducción a las Consideraciones finales) y Ley 25.326 pp. 1–20 (artículos 1 a 48). Se excluyen portadas, autoridades, índices, páginas en blanco, acrónimos, referencias y anexos, porque no aportan contenido para responder y agregarían ruido a la búsqueda. Las tres respuestas esperadas de las pruebas (Guía p. 29, Lineamientos p. 27, Ley p. 4) quedan dentro del recorte.

Los números de página son los **impresos** (empiezan en 1); LangChain los guarda en `metadata["page"]` desde 0, así que la conversión es `metadata["page"] + 1`.

In [4]:
MODELO_GENERADOR = os.getenv("OLLAMA_MODEL", "granite4:latest")
MODELO_EMBEDDINGS = os.getenv("OLLAMA_EMBEDDING_MODEL", "bge-m3")

# Cada documento: (ruta, página inicial, página final) con numeración impresa.
DOCUMENTOS = [
    (Path("datos/guia_ia_responsable_aaip_2024.pdf"), 6, 49),
    (Path("datos/lineamientos_plan_datos_cft_aaip_2023.pdf"), 9, 30),
    (Path("datos/ley_25326_texto_actualizado.pdf"), 1, 20),
]

for ruta, pagina_inicial, pagina_final in DOCUMENTOS:
    if not ruta.exists():
        raise FileNotFoundError(
            f"No encontré {ruta}. Copiá el PDF a la carpeta datos/ antes de seguir."
        )
    print(f"{ruta.name}: páginas {pagina_inicial} a {pagina_final}")

print("\nModelo generador:", MODELO_GENERADOR)
print("Modelo de embeddings:", MODELO_EMBEDDINGS)

guia_ia_responsable_aaip_2024.pdf: páginas 6 a 49
lineamientos_plan_datos_cft_aaip_2023.pdf: páginas 9 a 30
ley_25326_texto_actualizado.pdf: páginas 1 a 20

Modelo generador: granite4:latest
Modelo de embeddings: bge-m3


## 3. Cargar y revisar antes de indexar

Cargamos cada PDF completo, porque `PyPDFLoader` entrega una lista de páginas, y después nos quedamos solo con el rango de cada documento mediante `metadata["page"] + 1`. Todas las páginas del recorte quedan juntas en una lista, con el archivo en `source` y la página en `page`.

**Predicción.** 
Antes de ejecutar: 
1. ¿cuántas páginas esperás en total?: Espero 86 páginas, 44 de la Guía (6 a 49), 22 de los Lineamientos (9 a 30) y 20 de la Ley (1 a 20), contando la primera y la última de cada rango.
2. ¿Cuál creés que será más difícil de convertir a texto: una página de prosa o una con ilustración?: Creo que una página con ilustración será más difícil que una de prosa, porque PyPDFLoader solo lee la capa de texto del PDF y no convierte imágenes a texto.

In [5]:
paginas_del_recorte = []

for ruta, pagina_inicial, pagina_final in DOCUMENTOS:
    paginas_del_pdf = PyPDFLoader(str(ruta)).load()
    paginas_del_documento = []

    for pagina in paginas_del_pdf:
        pagina_impresa = pagina.metadata["page"] + 1

        if pagina_inicial <= pagina_impresa <= pagina_final:
            paginas_del_documento.append(pagina)

    print(
        f"{ruta.name}: {len(paginas_del_pdf)} páginas en el PDF, "
        f"{len(paginas_del_documento)} en el recorte"
    )
    assert len(paginas_del_documento) == pagina_final - pagina_inicial + 1
    paginas_del_recorte.extend(paginas_del_documento)

print("\nPáginas del recorte (3 documentos):", len(paginas_del_recorte))

guia_ia_responsable_aaip_2024.pdf: 62 páginas en el PDF, 44 en el recorte
lineamientos_plan_datos_cft_aaip_2023.pdf: 36 páginas en el PDF, 22 en el recorte
ley_25326_texto_actualizado.pdf: 21 páginas en el PDF, 20 en el recorte

Páginas del recorte (3 documentos): 86


In [6]:
def normalizar(texto):
    # El PDF corta líneas: se pasa a minúsculas y se unen los espacios.
    return " ".join(texto.lower().split())


# Caso, archivo, página impresa donde debería estar y evidencia esperada.
EVIDENCIAS = [
    ("R1", "guia_ia_responsable_aaip_2024.pdf", 29, "grandes volúmenes de datos"),
    ("R2", "lineamientos_plan_datos_cft_aaip_2023.pdf", 27, "altos estándares de seguridad"),
    ("R3", "ley_25326_texto_actualizado.pdf", 4, "libre, expreso e informado"),
]

for caso, archivo, pagina_esperada, evidencia in EVIDENCIAS:
    for pagina in paginas_del_recorte:
        es_la_pagina = (
            Path(pagina.metadata["source"]).name == archivo
            and pagina.metadata["page"] + 1 == pagina_esperada
        )
        if es_la_pagina:
            texto = pagina.page_content
            esta = normalizar(evidencia) in normalizar(texto)
            print(f"\n{'=' * 70}\n{caso} — {archivo}, página impresa {pagina_esperada}")
            print("Caracteres extraídos:", len(texto))
            print(f"Evidencia «{evidencia}» en el texto normalizado:", esta)
            print("-" * 70)
            print(texto[:800])


R1 — guia_ia_responsable_aaip_2024.pdf, página impresa 29
Caracteres extraídos: 2556
Evidencia «grandes volúmenes de datos» en el texto normalizado: True
----------------------------------------------------------------------
Evaluación de Impacto
La Evaluación de Impacto en la Protección de Datos en la etapa de diseño, es una herra-
mienta fundamental para identificar y mitigar riesgos asociados al tratamiento de datos 
personales en un sistema de Inteligencia Artificial desde el principio. 
Entre los riesgos que permite detectar está la reidentificación de personas, sesgos algorít-
micos, uso indebido de datos, entre otros, antes de que el sistema esté desarrollado. 
Por lo general aquellas organizaciones que deberían realizar una evaluación de impacto 
son las que tratan grandes volúmenes de datos, que realizan operaciones con datos sensi-
bles, o que realizan una indagación sistemática y exhaustiva de aspectos personales de las 
personas que se base en un tratamiento automatizado y

In [8]:
# Página de control: la Guía p. 30 es la única página del recorte con una imagen
# (Figura 3, matriz de riesgos). No responde R1: sirve para comprobar la predicción
# «prosa vs ilustración» y es la página del caso L1.
for pagina in paginas_del_recorte:
    es_la_pagina = (
        Path(pagina.metadata["source"]).name == "guia_ia_responsable_aaip_2024.pdf"
        and pagina.metadata["page"] + 1 == 30
    )
    if es_la_pagina:
        texto = pagina.page_content
        print("Guía p. 30 — página con ilustración (Figura 3: matriz de riesgos)")
        print("Caracteres extraídos:", len(texto))
        print("-" * 70)
        print(texto)

Guía p. 30 — página con ilustración (Figura 3: matriz de riesgos)
Caracteres extraídos: 2172
----------------------------------------------------------------------
Figura 3: Matriz de riesgos 
Guías y recomendaciones para la evaluación de impacto en un sistema de IA
Para mayor detalle, se recomienda tomar como referencia la evaluación de impacto ética 
publicada por la UNESCO, la que principalmente tiene como objetivo evaluar si los algoritmos 
se encuentran alineados con los valores, principios y orientaciones establecidos por su Reco-
mendación. Esta evaluación debería garantizar la transparencia siendo abierta al público la 
información sobre estos sistemas y cómo se desarrollaron.
Oportunamente, la AAIP de Argentina y la Unidad Reguladora y de Control de Datos Personales de 
Uruguay, publicaron una Guía “Evaluación de Impacto en la Protección de Datos”. Este instrumen-
to no fue desarrollado específicamente para la evaluación de impacto de sistemas que utilicen la 
Inteligencia Art

**Observación de la extracción.** Se comparó cada página con el PDF abierto en la misma página impresa. El contenido de las tres páginas de prosa coincide con el PDF; el ruido que aparece es:

- **Guía (R1, p. 29):** el encabezado («Guía para entidades públicas y privadas…») y el pie propio del PDF («29 | Recomendaciones de Transparencia…») se extraen, pero desplazados al final del texto de la página. El pie con logos de la institución es una imagen y no se extrae. Hay palabras cortadas con guion («herra- mienta», «sensi- bles»).
- **Lineamientos (R2, p. 27):** texto limpio; el número de página aparece al inicio y hay algún corte de palabra («su - puesto»).
- **Ley (R3, p. 4):** conserva el encabezado y el pie del navegador (fecha, título, URL, «4/21») al final de la página.

Este ruido no se limpió: se declara como límite del corpus.


**Observación de predicciones.** 

1. Se cargaron 86 páginas, como esperaba. El assert confirma el recorte exacto de cada documento.

2. Las páginas de prosa (Guía p. 29, Lineamientos p. 27, Ley p. 4) se extraen completas y coinciden con el PDF, con defectos menores: palabras cortadas con guion en la Guía («sensi- bles») y encabezados repetidos en la Guía y la Ley. La página con ilustración (Guía p. 30, Figura 3) confirma la predicción: la matriz de riesgos es una imagen y su contenido (ejes, valores y colores) no se extrae; solo queda la leyenda como texto suelto. Por eso L1 no debería poder responderse.